<a href="https://colab.research.google.com/github/abdulhaseeb941/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("logged in")

logged in


In [4]:
from huggingface_hub import list_repo_files

files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
print(len(files), "files")
for f in files[:30]:
    print(f)

24 files
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_perform

In [5]:
import pandas as pd
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
)
mar = pd.read_parquet(path)
print(mar.shape)
print(list(mar.columns))
print(mar.head(3).to_string())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

(9841378, 30)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
  report_date           client_hash_id           content_hash_id  client_has_gsc  client_has_ga4  gsc_data_available ga4_data_available  gsc_impressions  gsc_clicks  gsc_sum_position  gsc_avg_position  ga4_pageviews  ga4_sessions  ga4_users  ga4_engaged_sessions  ga4_total_engagement_sec  sessions_organic  sessions_direct  sessions_referral  sessions_social  sessions_paid  sessions_ai  ai_chatgpt  ai_perplexity  ai_gemini  ai_copilot  ai_claude  ai_meta  a

In [6]:
keys = ["report_date", "client_hash_id", "content_hash_id"]
dupes = mar.duplicated(subset=keys).sum()
print("rows:", len(mar))
print("duplicate key rows:", dupes)
print("date span:", mar["report_date"].min(), "to", mar["report_date"].max())

rows: 9841378
duplicate key rows: 0
date span: 2026-03-01 to 2026-03-31


In [7]:
avail = mar["gsc_data_available"].fillna(False).astype(bool)
print("rows with gsc_data_available true:", avail.sum())
print("share kept:", round(avail.mean() * 100, 2), "%")
print("rows dropped (false or missing):", (~avail).sum())

rows with gsc_data_available true: 3611061
share kept: 36.69 %
rows dropped (false or missing): 6230317


In [8]:
import pandas as pd
from huggingface_hub import hf_hub_download

def monthly(month):
    p = hf_hub_download("FlyRank/internship-warehouse",
        f"fact_content_daily_performance/month={month}/data_0.parquet",
        repo_type="dataset")
    d = pd.read_parquet(p, columns=["client_hash_id","content_hash_id",
        "gsc_data_available","gsc_clicks","gsc_impressions"])
    d = d[d["gsc_data_available"].fillna(False).astype(bool)]
    return d.groupby(["client_hash_id","content_hash_id"])[["gsc_clicks","gsc_impressions"]].sum()

feb, mar_m, apr = monthly("2026-02"), monthly("2026-03"), monthly("2026-04")
t = feb.join(mar_m, lsuffix="_feb", rsuffix="_mar", how="inner").join(apr.add_suffix("_apr"), how="left")
t["gsc_clicks_apr"] = t["gsc_clicks_apr"].fillna(0)

declined = (t["gsc_clicks_feb"] >= 30) & (t["gsc_clicks_mar"] <= 0.7 * t["gsc_clicks_feb"])
d = t[declined].copy()
d["label"] = (d["gsc_clicks_apr"] >= 0.8 * d["gsc_clicks_feb"]).astype(int)

print("items with data in Feb and Mar:", len(t))
print("declined items (rows in your dataset):", len(d))
print("label counts:"); print(d["label"].value_counts())

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 89.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/tmp/ipykernel_6312/4044162730.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d = d[d["gsc_data_available"].fillna(False).astype(bool)]


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

items with data in Feb and Mar: 134238
declined items (rows in your dataset): 793
label counts:
label
0    685
1    108
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X = pd.DataFrame({
    "clicks_feb": d["gsc_clicks_feb"],
    "clicks_mar": d["gsc_clicks_mar"],
    "drop_ratio": d["gsc_clicks_mar"] / d["gsc_clicks_feb"],
    "impr_mar": d["gsc_impressions_mar"],
    "ctr_mar": d["gsc_clicks_mar"] / d["gsc_impressions_mar"].replace(0, float("nan")),
}).fillna(0)
y = d["label"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
m = RandomForestClassifier(n_estimators=200, random_state=0, class_weight="balanced").fit(Xtr, ytr)
print("honest AUC:", round(roc_auc_score(yte, m.predict_proba(Xte)[:,1]), 3))

honest AUC: 0.688


In [10]:
# 1) THE TRAP: add a label-derived column on purpose (April clicks = the future)
X_leak = X.copy()
X_leak["clicks_apr"] = d["gsc_clicks_apr"]

Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leak, y, test_size=0.3, random_state=0, stratify=y)
m_leak = RandomForestClassifier(n_estimators=200, random_state=0, class_weight="balanced").fit(Xtr_l, ytr_l)
print("LEAKY AUC (with clicks_apr):", round(roc_auc_score(yte_l, m_leak.predict_proba(Xte_l)[:,1]), 3))

# 2) DELETE IT and keep the honest number
X_leak = X_leak.drop(columns=["clicks_apr"])
print("columns after delete:", list(X_leak.columns))
print("honest AUC (back to normal):", round(roc_auc_score(yte, m.predict_proba(Xte)[:,1]), 3))

LEAKY AUC (with clicks_apr): 0.972
columns after delete: ['clicks_feb', 'clicks_mar', 'drop_ratio', 'impr_mar', 'ctr_mar']
honest AUC (back to normal): 0.688
